## Part 2 - Data Preprocessing

This notebook prepares the Nasdaq dataset for modeling by:
1. Loading the raw multi-ticker CSV
2. Applying feature engineering from `src/data/preprocess.py`
3. Validating output schema and missing values
4. Saving a processed CSV for training

### 1. MULTI-FEATURE EXTENSION

### Why Use Multiple Features Instead of Just Raw Price?

A naive approach to stock price prediction uses only the **closing price** as model input. This has two fundamental limitations:

1. **Non-stationarity**: Raw prices drift over long periods — AAPL at \$5 in 2000 and \$180 in 2024 represent the same stock, but their absolute values are not comparable on the same numeric scale. A model trained on raw prices must implicitly account for these magnitude differences across the entire 10-year window, wasting representational capacity on scale rather than pattern.

2. **Loss of market context**: Price alone does not encode *how* or *why* a price is at a given level. Two stocks sitting at identical price levels can have completely different momentum dynamics, volatility regimes, and trend states. The model cannot distinguish these without additional signals.

#### Feature Engineering Philosophy

We enrich each daily observation with a set of **complementary technical indicators**, grouped by the dimension of market information they capture:

| Category | Features | What they encode |
|---|---|---|
| **Price dynamics** | `log_return` | Day-over-day percentage change — a stationary, scale-free proxy for price movement |
| **Momentum** | `rsi`, `macd`, `macd_signal`, `macd_hist` | Rate, direction, and acceleration of recent price change |
| **Volatility** | `atr`, `bb_upper`, `bb_middle`, `bb_lower` | Daily price dispersion and distance from statistical price bands |
| **Trend** | `ema_10`, `ema_20`, `ema_50` | Multi-horizon smoothed price levels and their cross-relationships |
| **Adjusted price** | `adj_close` | Split- and dividend-adjusted price for long-horizon comparability |

The LSTM/GRU model receives all of these at every timestep, allowing it to learn **which features are predictive under which market conditions** — a form of implicit feature selection that is impossible with price alone.

> **On retaining raw OHLC columns**: Despite adding derived features, the original Open, High, Low, Close, and Volume columns are preserved. The model's prediction target is the actual closing price; raw price columns serve as both input context and the reference frame for inverse-transforming scaled predictions back to their original USD values.

In [ ]:
import sys  # handle with python interpreter
from pathlib import Path # Handle file paths
import pandas as pd

ROOT = Path.cwd().resolve().parent # Find the root directory, resolve() to get absolute path, parent to go up one level
# Instead of /DL4AI-240166-project-1/notebooks, we want /DL4AI-240166-project-1
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Now Python now how to get the src/data/preprocess.py file
from src.data.preprocess import add_all_features

In [9]:
raw_path = Path("/Users/cps/DL4AI-240166-project-1/notebooks/data/nasdaq/csv/tech_nasdaq_stock_data.csv")
df = pd.read_csv(raw_path)

df.columns = [c.lower() for c in df.columns]
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

print("Raw shape:", df.shape)
display(df.head(3))

Raw shape: (23130, 8)


,date,open,high,low,close,adj_close,volume,ticker
0,2016-01-04,25.652500,26.342501,25.500000,26.337500,23.730949,270597600,AAPL
1,2016-01-05,26.437500,26.462500,25.602501,25.677500,23.136265,223164000,AAPL
2,2016-01-06,25.139999,25.592501,24.967501,25.174999,22.683498,273829600,AAPL


In [ ]:
frames = [] # Create an empty list to store the processed data frames

# Run the loop
#df.groupby("ticker", sort = False) Divide the data into groups based on the ticker column
# for ticker, group means for each iteration, get the rows with the same ticker
for ticker, group in df.groupby("ticker", sort=False):
    g = group.copy() # Not modifying the original group
    g = g.drop(columns=["ticker"])          # keep only market columns
    g = g.set_index("date").sort_index()

    g = add_all_features(g)                   # compute indicators/features
    # Defensive cleanup for old cached outputs that may still include this column.
    if "adj_close_fwd" in g.columns:
        g = g.drop(columns=["adj_close_fwd"])

    g["ticker"] = ticker                     # add ticker back
    frames.append(g.reset_index())

df_features = pd.concat(frames, ignore_index=True) # ignore_index=True to avoid index conflicts

print("Feature shape:", df_features.shape)
print("Tickers:", df_features["ticker"].nunique())
display(df_features.head(3))

Feature shape: (22689, 20)
Tickers: 9


,date,open,high,low,close,adj_close,volume,log_return,rsi,macd,macd_signal,macd_hist,atr,ema_10,ema_20,ema_50,bb_upper,bb_middle,bb_lower,ticker
0,2016-03-15,25.990000,26.295000,25.962500,26.145000,23.685328,160270800,0.019894,67.377844,0.384964,0.249910,0.135054,0.516683,25.419239,25.049804,24.913412,26.286548,24.859875,23.433202,AAPL
1,2016-03-16,26.152500,26.577499,26.147499,26.492500,24.000132,153214000,0.013204,70.323953,0.449557,0.289839,0.159717,0.510670,25.614377,25.187203,24.975337,26.541733,24.958000,23.374267,AAPL
2,2016-03-17,26.379999,26.617500,26.240000,26.450001,23.961632,137682800,-0.001605,69.497309,0.491650,0.330202,0.161449,0.501157,25.766309,25.307470,25.033167,26.731654,25.077250,23.422846,AAPL


### Feature Engineering Summary

The table below documents every column produced by `add_all_features()`, its mathematical basis, and the justification for its inclusion in the model input.

| Feature | Formula / Period | Category | Justification |
|---|---|---|---|
| `log_return` | log(close_t / close_{t−1}) | Price dynamics | Converts non-stationary price levels to stationary log-scale returns; values are scale-free and comparable across different stocks and time periods, improving model generalisation |
| `rsi` | RSI(14) | Momentum | Oscillator bounded [0, 100]; values above 70 signal overbought conditions, below 30 signal oversold; provides a discrete "market state" the model cannot infer from raw price alone. The 14-day period is the Wilder standard, widely adopted across institutional trading systems |
| `macd` | EMA(12) − EMA(26) | Momentum | Positive MACD indicates short-term momentum is stronger than long-term trend (bullish); negative indicates bearish pressure; directly encodes the direction of trend change |
| `macd_signal` | EMA(9) of MACD | Momentum | Smoothed version of the MACD line; MACD–Signal crossovers are among the most widely used entry and exit signals in technical analysis — the model can implicitly learn these crossover patterns from the raw values |
| `macd_hist` | MACD − Signal | Momentum | Measures the divergence rate between MACD and its signal line; highlights momentum acceleration and deceleration. Note: `macd`, `macd_signal`, and `macd_hist` are mathematically related (hist = macd − signal), making all three partially redundant. However, each representation emphasises a different aspect — absolute momentum, smoothed trend, and divergence rate — and an attention-based model can learn to down-weight redundant dimensions |
| `atr` | ATR(14) | Volatility | Average True Range measures the average daily price range over the past 14 sessions; higher ATR signals wider expected price swings and elevated uncertainty |
| `ema_10` | EMA(10) | Trend | Short-horizon smoothed price; responsive to recent changes; crossover with longer EMAs signals short-term trend shifts |
| `ema_20` | EMA(20) | Trend | Medium-horizon smoothed price; approximately one trading month |
| `ema_50` | EMA(50) | Trend | Long-horizon smoothed price; approximately one trading quarter; EMA10/EMA50 crossover is a classic trend-following signal |
| `bb_upper` | SMA(20) + 2σ | Volatility | Upper Bollinger Band; price touching this level suggests potential overextension to the upside |
| `bb_middle` | SMA(20) | Volatility | Rolling 20-day mean; acts as a dynamic support/resistance level |
| `bb_lower` | SMA(20) − 2σ | Volatility | Lower Bollinger Band; price touching this level suggests potential oversold extension |
| `adj_close` | Split- and dividend-adjusted close | Price reference | Ensures historical price comparability over the full 10-year window (2016–2026); prevents artificial discontinuities at stock split dates that would otherwise appear as spurious price jumps to the model |

> **Target variable note**: The prediction target throughout Tasks 1 and 2 is `adj_close` for Nasdaq (adjusted for corporate actions) and `close` for Vietnam stocks (where no adjusted price is available from `vnstock`). The `compute_target_price()` utility in `src/data/preprocess.py` selects the appropriate column automatically.

In [11]:
print("Columns:")
print(df_features.columns.tolist())

print("\nTop missing-value counts:")
print(df_features.isna().sum().sort_values(ascending=False).head(10))

print("\nRows per ticker:")
print(df_features.groupby("ticker").size().sort_values(ascending=False))

Columns:
['date', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'log_return', 'rsi', 'macd', 'macd_signal', 'macd_hist', 'atr', 'ema_10', 'ema_20', 'ema_50', 'bb_upper', 'bb_middle', 'bb_lower', 'ticker']

Top missing-value counts:
date         0
open         0
bb_lower     0
bb_middle    0
bb_upper     0
ema_50       0
ema_20       0
ema_10       0
atr          0
macd_hist    0
dtype: int64

Rows per ticker:
ticker
AAPL     2521
AMZN     2521
GOOGL    2521
META     2521
MSFT     2521
MU       2521
NFLX     2521
NVDA     2521
QCOM     2521
dtype: int64


In [12]:
out_path = Path("data/nasdaq/csv/tech_nasdaq_stock_data_features.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

df_features.to_csv(out_path, index=False)
print(f"Saved to: {out_path}")

Saved to: data/nasdaq/csv/tech_nasdaq_stock_data_features.csv


In [19]:
print(df_features)

            date        open        high         low       close   adj_close  \
0     2016-03-15   25.990000   26.295000   25.962500   26.145000   23.685328   
1     2016-03-16   26.152500   26.577499   26.147499   26.492500   24.000132   
2     2016-03-17   26.379999   26.617500   26.240000   26.450001   23.961632   
3     2016-03-18   26.584999   26.625000   26.297501   26.480000   23.988815   
4     2016-03-21   26.482500   26.912500   26.285000   26.477501   23.986546   
...          ...         ...         ...         ...         ...         ...   
22684 2026-03-18  130.729996  132.740005  129.929993  130.470001  130.470001   
22685 2026-03-19  129.139999  132.679993  128.899994  131.279999  131.279999   
22686 2026-03-20  131.309998  132.750000  129.779999  129.899994  129.899994   
22687 2026-03-23  133.149994  133.970001  127.410004  128.350006  128.350006   
22688 2026-03-24  128.339996  129.179993  127.309998  128.669998  128.669998   

          volume  log_return        rsi